# Gradio: インターフェースからチャットボットへ

このチュートリアルでは **Gradio** について学びます。Gradio は、機械学習モデル、API、または任意の Python 関数のためのインタラクティブな Web アプリを簡単に構築・共有できる強力なオープンソースの Python ライブラリです。

このノートブックでは、Gradio の核となるコンセプトを迅速に習得します。以下をカバーします。

- **セットアップ**: Gradio のインストールと `.env` ファイルからの API キーの読み込み。
- **`gr.Interface`**: 関数から UI を作成する主要な高レベルクラス。
- **コンポーネント**: テキストボックス、スライダー、画像フィールドなど、さまざまな入出力要素。
- **`gr.Blocks`**: より複雑でカスタムなレイアウトを作成するための低レベル API。
- **`gr.ChatInterface`**: チャットボットを構築するための特化した使いやすいインターフェース。
- **共有**: 単一のコマンドでアプリの公開リンクを生成する方法。

## 1. セットアップ

まず、`gradio` と `python-dotenv` をインストールします。`python-dotenv` は API キーを安全に管理するために使用します。

In [15]:
# Uncomment the following line to install necessary packages
!pip install -q gradio python-dotenv -q

In [2]:
import gradio as gr
import os
from dotenv import load_dotenv

# Load environment variables from .env file
load_dotenv()

openai_api_key = os.getenv("OPENAI_API_KEY")

print("API keys loaded!")

API keys loaded!


## 2. `gr.Interface` クラス: Gradio の核心

UI を作成する最も簡単な方法は `gr.Interface` クラスです。これは、あらゆる Python 関数をインタラクティブなインターフェースでラップするように設計されています。3 つの核心引数が必要です。

- `fn`: ラップする関数。
- `inputs`: ユーザー向けの入力コンポーネント（例: `"text"`、`"image"`、`"slider"`）。
- `outputs`: 結果を表示する出力コンポーネント。

簡単な "Hello World" から始めましょう。

In [4]:
def greet(name):
    return f"Hello, {name}!"

demo = gr.Interface(
    fn=greet, 
    inputs=gr.Textbox(label="Name", placeholder="Enter your name here..."), 
    outputs="text"
)

demo.launch()

* Running on local URL:  http://127.0.0.1:7861
* To create a public link, set `share=True` in `launch()`.


Created dataset file at: .gradio/flagged/dataset1.csv


![Gradio の UI](/gradio.webp)

### 複数の入出力の処理

Gradio は、複数の入力と出力を持つ関数も処理できます。`inputs` と `outputs` 引数にコンポーネントのリストを指定するだけです。順序が重要です！

In [5]:
def calculate(num1, num2, operation):
    if operation == "add":
        result = num1 + num2
    elif operation == "subtract":
        result = num1 - num2
    elif operation == "multiply":
        result = num1 * num2
    elif operation == "divide":
        if num2 == 0:
            return "Error", "Cannot divide by zero"
        result = num1 / num2
    return result, f"Performed {operation}."

demo = gr.Interface(
    fn=calculate,
    inputs=[
        gr.Number(label="Number 1"),
        gr.Number(label="Number 2"),
        gr.Dropdown(["add", "subtract", "multiply", "divide"], label="Operation")
    ],
    outputs=[
        gr.Textbox(label="Result"),
        gr.Textbox(label="Status")
    ],
    title="Simple Calculator",
    description="A simple calculator to demonstrate multiple inputs and outputs.",
    examples=[[5, 3, "add"], [10, 2, "divide"]]
)

demo.launch()

* Running on local URL:  http://127.0.0.1:7862
* To create a public link, set `share=True` in `launch()`.


## 3. `gr.Blocks`: カスタムレイアウト用

UI レイアウトをより細かく制御する必要がある場合、`gr.Blocks` を使用します。これにより、行、列、タブを使ってコンポーネントを配置し、より複雑なイベント駆動型のインタラクションを定義できます。

ここでは、2 つのタブを持つ簡単なアプリを作成します。ボタンの `.click()` メソッドがコンポーネントを接続していることに注目してください。

In [6]:
import numpy as np

def flip_image(img):
    return np.fliplr(img)

def greet_user(name):
    return f"How are you, {name}?"

with gr.Blocks(theme=gr.themes.Soft()) as demo:
    gr.Markdown("# Custom UI with Blocks")
    with gr.Tabs():
        with gr.TabItem("Image Flipper"):
            with gr.Row():
                image_input = gr.Image()
                image_output = gr.Image()
            image_button = gr.Button("Flip Image")
            
        with gr.TabItem("Greeter"):
            name_input = gr.Textbox(label="Name")
            greeting_output = gr.Textbox(label="Greeting")
            greet_button = gr.Button("Greet")

    # Define interactions
    image_button.click(fn=flip_image, inputs=image_input, outputs=image_output)
    greet_button.click(fn=greet_user, inputs=name_input, outputs=greeting_output)

demo.launch()

* Running on local URL:  http://127.0.0.1:7863
* To create a public link, set `share=True` in `launch()`.


## 4. `gr.ChatInterface` でチャットボットを構築する

Gradio はチャットボットの作成に優れています。`gr.ChatInterface` は、単一の関数から完全なチャットボット UI を作成する高レベルな抽象化です。

関数は 2 つの引数、`message` と `history` を受け取る必要があります。`history` は会話の状態を保持します。

簡単なチャットボットを作成しましょう。

In [ ]:
import openai

def openai_chatbot(message, history):
    messages = []
    # Convert history to OpenAI chat format
    for user, assistant in history:
        messages.append({"role": "user", "content": user})
        messages.append({"role": "assistant", "content": assistant})
    messages.append({"role": "user", "content": message})
    try:
        client = openai.OpenAI(api_key=openai_api_key)
        response = client.chat.completions.create(
            model="gpt-4o-mini",
            messages=messages,
            stream=True
        )
        partial = ""
        for chunk in response:
            delta = chunk.choices[0].delta.content or ""
            partial += delta
            yield partial
    except Exception as e:
        yield f"[Error] {str(e)}"

demo = gr.ChatInterface(
    fn=openai_chatbot,
    title="OpenAI GPT-4o-mini Chatbot",
    description="A chatbot powered by OpenAI's GPT-4o-mini model.",
    examples=["Hello!", "What is Gradio?", "Tell me a joke."]
)

demo.launch()

/Users/devon/.pyenv/versions/3.10.14/lib/python3.10/site-packages/gradio/chat_interface.py:345: UserWarning: The 'tuples' format for chatbot messages is deprecated and will be removed in a future version of Gradio. Please set type='messages' instead, which uses openai-style 'role' and 'content' keys.
  self.chatbot = Chatbot(


* Running on local URL:  http://127.0.0.1:7869
* To create a public link, set `share=True` in `launch()`.


![Gradio チャットボット](/gradio2.webp)

## 5. アプリの共有

Gradio の最も強力な機能の 1 つは、アプリの公開可能で共有可能なリンクを作成できることです。これにより、誰でもブラウザからデモを試すことができ、コードはあなたのマシン（またはスクリプトを実行している場所）で実行されます。

これを有効にするには、`launch()` メソッドに `share=True` を追加するだけです。

**注**: リンクは一時的なもの（通常 72 時間）で、ローカルサーバーへの直接トンネルを提供します。信頼できる人とのみ共有してください。

In [ ]:
# This is a non-blocking cell to show the syntax.
# To run it, you would call demo.launch(share=True)
# For example:
# 
# def hello(name):
#    return f"Hi {name}!"
# 
# demo = gr.Interface(fn=hello, inputs="text", outputs="text")
# demo.launch(share=True) # This will generate a public URL like https://....gradio.live

# ノートブックのまとめ

このノートブックは、機械学習デモやチャットボットを迅速にプロトタイピングするための Python ライブラリである Gradio を使って、インタラクティブな Web アプリを構築する実践チュートリアルです。以下をカバーします。

- **セットアップ:** Gradio のインストールと API キーの安全な読み込み。
- **gr.Interface:** Python 関数のためのシンプルな UI の作成。
- **複数の入出力:** 複数の引数と戻り値を持つ関数の処理。
- **gr.Blocks:** カスタムレイアウトとマルチタブアプリの構築。
- **gr.ChatInterface:** OpenAI の GPT-4o-mini モデルを含むチャットボット UI の構築。
- **Web と公開共有:** ブラウザでアプリにアクセスし、他の人と共有する方法。

最終的に、以下のことができるようになります。
- ノートブックまたは Web アプリとして Gradio アプリを構築・起動する
- 実際の LLM（OpenAI、Hugging Face など）に接続する
- インタラクティブなデモを簡単に他の人と共有する